# DATA-445: In-Class Assignment 12

## Overview
Consider the `framingham.csv` data file. The dataset is available on the Kaggle website, and it is from an ongoing cardiovascular study on residents of the town of Framingham, Massachusetts. The classification goal is to predict whether the patient has 10-year risk of future coronary heart disease (CHD). The dataset provides the patients' information. It includes over 4,000 records and 15 attributes. Each attribute is a potential risk factor. There are both demographic, behavioral and medical risk factors.

## Data Attributes

### Demographic:
- **Sex**: male or female (Nominal)
- **Age**: Age of the patient (Continuous - Although the recorded ages have been truncated to whole numbers, the concept of age is continuous)

### Behavioral:
- **Current Smoker**: whether or not the patient is a current smoker (Nominal)
- **Cigs Per Day**: the number of cigarettes that the person smoked on average in one day (can be considered continuous as one can have any number of cigarettes, even half a cigarette)

### Medical (history):
- **BP Meds**: whether or not the patient was on blood pressure medication (Nominal)
- **Prevalent Stroke**: whether or not the patient had previously had a stroke (Nominal)
- **Prevalent Hyp**: whether or not the patient was hypertensive (Nominal)
- **Diabetes**: whether or not the patient had diabetes (Nominal)

### Medical (current):
- **Tot Chol**: total cholesterol level (Continuous)
- **Sys BP**: systolic blood pressure (Continuous)
- **Dia BP**: diastolic blood pressure (Continuous)
- **BMI**: Body Mass Index (Continuous)
- **Heart Rate**: heart rate (Continuous - In medical research, variables such as heart rate though in fact discrete, yet are considered continuous because of large number of possible values)
- **Glucose**: glucose level (Continuous)

### Target Variable:
- **TenYearCHD**: 10 year risk of coronary heart disease CHD (binary: "1" means "Yes", "0" means "No")

## Exercise 1: Data Loading and Preprocessing (4 points)

Using the `pandas` library, read the csv data file and create a data-frame called `heart`. Remove the missing values.

In [6]:
import pandas as pd

heart = pd.read_csv('framingham.csv')
heart.head()

heart.dropna().reset_index(drop=True)
heart.head()

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


## Exercise 2: Define Features and Cross-Validation Strategy (4 points)

Define `age`, `currentSmoker`, `totChol`, `BMI`, and `heartRate` as the input variables, and `TenYearCHD` is the target variable. Define a 5-fold cross-validation strategy.

In [8]:
# Your code here

from sklearn.model_selection import KFold

X = heart[["age", "currentSmoker", "totChol", "BMI", "heartRate"]].copy()
y = heart["TenYearCHD"].copy()

skf = KFold(n_splits=5, shuffle=True, random_state=42)

## Exercise 3: AdaBoostClassifier Evaluation (8 points)

Train and evaluate an `AdaBoostClassifier` model (with `n_estimators = 500`, `max_depth = 3`, and `learning_rate = 0.01`) over the 5-fold cross-validation strategy defined in Exercise 2. Using 10% as cutoff value, report the average recall over the 5-folds.

In [10]:
# Your code here
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import recall_score
from sklearn.impute import SimpleImputer

ada_recall = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)
    
    ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=3),
                             n_estimators=500,
                             learning_rate=0.01,
                             random_state=42)
    ada.fit(X_train, y_train)
    
    y_pred = ada.predict_proba(X_test)[:, 1]
    y_pred = (y_pred > 0.1).astype(int)
    
    score = recall_score(y_test, y_pred)
    ada_recall.append(score)
    print(f"Fold {i+1} Recall: {score}")

print(f"Average Recall: {sum(ada_recall)/len(ada_recall)}")

Fold 1 Recall: 1.0
Fold 2 Recall: 1.0
Fold 3 Recall: 1.0
Fold 4 Recall: 1.0
Fold 5 Recall: 1.0
Average Recall: 1.0


## Exercise 4: RandomForestClassifier Evaluation (8 points)

Train and evaluate a `RandomForestClassifier` model (with `n_estimators = 500` and `max_depth = 3`) over the 5-fold cross-validation strategy defined in Exercise 2. Using 10% as cutoff value, report the average recall over the 5-folds.

In [11]:
# Your code here
from sklearn.ensemble import RandomForestClassifier

rf_recall = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    rf = RandomForestClassifier(n_estimators=500, 
                                max_depth=3, 
                                random_state=42,
                                n_jobs=-1)
    rf.fit(X_train, y_train)
    
    y_pred = rf.predict_proba(X_test)[:, 1]
    y_pred = (y_pred > 0.1).astype(int)
    
    score = recall_score(y_test, y_pred)
    rf_recall.append(score)
    print(f"Fold {i+1} Recall: {score}")

print(f"Average Recall: {sum(rf_recall)/len(rf_recall)}")

Fold 1 Recall: 0.8629032258064516
Fold 2 Recall: 0.8740157480314961
Fold 3 Recall: 0.9140625
Fold 4 Recall: 0.8613138686131386
Fold 5 Recall: 0.8671875
Average Recall: 0.8758965684902172


## Exercise 5: ExtraTreesClassifier Evaluation (8 points)

Train and evaluate an `ExtraTreesClassifier` model (with `n_estimators = 500` and `max_depth = 3`) over the 5-fold cross-validation strategy defined in Exercise 2. Using 10% as cutoff value, report the average recall over the 5-folds.

In [12]:
# Your code here

from sklearn.ensemble import ExtraTreesClassifier

et_recall = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    et = ExtraTreesClassifier(n_estimators=500, 
                               max_depth=3,
                               random_state=42,
                               n_jobs=-1)
    et.fit(X_train, y_train)
    
    y_pred = et.predict_proba(X_test)[:, 1]
    y_pred = (y_pred > 0.1).astype(int)
    
    score = recall_score(y_test, y_pred)
    et_recall.append(score)
    print(f"Fold {i+1} Recall: {score}")

print(f"Average Recall: {sum(et_recall)/len(et_recall)}")

Fold 1 Recall: 1.0
Fold 2 Recall: 0.9921259842519685
Fold 3 Recall: 1.0
Fold 4 Recall: 0.9927007299270073
Fold 5 Recall: 1.0
Average Recall: 0.9969653428357951


## Exercise 6: Model Comparison and Selection (3 points)

Using the results from Exercises 3, 4, and 5, what model would you use to predict `TenYearCHD` based on the recall? Justify your choice.

**Your answer here:**

I would use 'adaboost' to predict TenYearCHD because the recall score is the highest.